In [79]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [80]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [81]:
# Removing Unnecessary features
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"], axis=1)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [82]:
# Label encoding "Gender" Feature
le = LabelEncoder()
df["Gender"] = le.fit_transform(df["Gender"])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [83]:
# OneHot encoding the "Geography" feature
ohe = OneHotEncoder()
encoded = ohe.fit_transform(df[["Geography"]]).toarray()
feature_names = ohe.get_feature_names_out(["Geography"])
encoded_df = pd.DataFrame(encoded, columns=feature_names)

# Replace original column
df=pd.concat([df.drop('Geography',axis=1),encoded_df],axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [84]:
# Saving the preprocessing pickle file
with open("labelEncoding_gender.pkl", "wb") as file:
    pickle.dump(le, file)

with open("OneHotEncoding_geo.pkl", "wb") as file:
    pickle.dump(ohe, file)

In [85]:
# Splitting independent and dependent feature
independent_features = df.drop(columns="Exited", axis=1)
dependent_feature = df["Exited"]

# Split the data into training and testing data
x_train, x_test, y_train, y_test = train_test_split(independent_features, dependent_feature, test_size=0.2, random_state=42)

# Scale the train and test input data
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [86]:
# Saving scaler as pickle file
with open("Scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

### ANN IMPLEMENTATION

In [87]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [88]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(independent_features.shape[1], )),       # Hidden Layer 1 connected to input layer 
    Dense(32, activation='relu'),                                                      # Hidden Layer 2 
    Dense(1, activation='sigmoid')                                                     # Output Layer
]
)

c:\Users\manju\Gen AI\GA\Lib\site-packages\keras\src\layers\core\dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [89]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [90]:
# Compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [91]:
import datetime
# setup the tensorboard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(log_dir, histogram_freq=1)


In [92]:
# setup early stopping
early_stopping_callbacks = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [93]:
# Train the model
history = model.fit(
    x_train, y_train, validation_data=(x_test, y_test), epochs=100, 
    callbacks=[tensorflow_callback, early_stopping_callbacks]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7847 - loss: 0.4986 - val_accuracy: 0.8320 - val_loss: 0.3953
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8362 - loss: 0.4007 - val_accuracy: 0.8565 - val_loss: 0.3559
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8528 - loss: 0.3591 - val_accuracy: 0.8600 - val_loss: 0.3468
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8556 - loss: 0.3470 - val_accuracy: 0.8595 - val_loss: 0.3470
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.8614 - loss: 0.3406 - val_accuracy: 0.8625 - val_loss: 0.3419
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8620 - loss: 0.3325 - val_accuracy: 0.8635 - val_loss: 0.3394
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8611 - loss: 0.3331 - val_accuracy: 0.8660 - val_loss: 0.3385
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8633 - loss: 0.3292 - val_acc

In [94]:
# Save the model
model.save("model.h5")

In [95]:
# Load the tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [96]:
%tensorboard --logdir logs/fit/20250704-075147/train/

Reusing TensorBoard on port 6006 (pid 8648), started 3:56:50 ago. (Use '!kill 8648' to kill it.)